# ABC Ltd. — Employee Attrition Predictor
### Logistic Regression Model for Managerial Decision Support

This notebook builds a logistic regression model that predicts the probability an employee will leave ABC Ltd., based on HR data. It is designed to run top-to-bottom in Google Colab.

**Steps covered:**
1. Upload & load the dataset
2. Clean and encode the data
3. Train/test split
4. Scale features
5. Train a logistic regression model
6. Evaluate the model
7. Interpret which factors drive attrition
8. Export the model for deployment

## Step 1: Upload the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # select ABC_Ltd_Employee_Attrition.csv when prompted

## Step 2: Load and inspect the data

In [ ]:
import pandas as pd

df = pd.read_csv("ABC_Ltd_Employee_Attrition.csv")
print(df.shape)
df.head()

In [ ]:
df.info()
df['Attrition'].value_counts()

## Step 3: Clean up — drop useless/constant columns
`EmployeeCount`, `StandardHours`, and `Over18` are constant across all rows (no predictive value). `EmployeeNumber` is just an ID.

In [ ]:
df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'])

## Step 4: Encode the target and categorical variables

In [ ]:
# Target: Yes/No -> 1/0
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# One-hot encode remaining categorical columns
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print(categorical_cols)

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.shape

## Step 5: Split features and target

In [ ]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Attrition'])
y = df_encoded['Attrition']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

## Step 6: Scale the features
Logistic regression converges better and coefficients are more comparable when features are scaled.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 7: Train the logistic regression model

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, class_weight='balanced')
model.fit(X_train_scaled, y_train)

`class_weight='balanced'` matters here — only ~16% of employees have `Attrition = Yes`, so without it the model leans toward predicting "No" for everyone.

## Step 8: Evaluate the model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

## Step 9: Interpret which features drive attrition (for your managerial report)

In [ ]:
import numpy as np

coefs = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', ascending=False)

print("Top factors INCREASING attrition risk:")
print(coefs.head(10))

print("\nTop factors DECREASING attrition risk:")
print(coefs.tail(10))

In [ ]:
import matplotlib.pyplot as plt

top_features = pd.concat([coefs.head(8), coefs.tail(8)])
plt.figure(figsize=(8,6))
plt.barh(top_features['feature'], top_features['coefficient'])
plt.xlabel("Coefficient (impact on attrition log-odds)")
plt.title("What Drives Employee Attrition")
plt.tight_layout()
plt.show()

## Step 10: Save the model and scaler for deployment

In [ ]:
import joblib

joblib.dump(model, 'attrition_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(list(X.columns), 'feature_columns.pkl')  # column order matters for new predictions

files.download('attrition_model.pkl')
files.download('scaler.pkl')
files.download('feature_columns.pkl')

## Step 11: Test a single prediction (simulating what your deployed tool will do)

In [ ]:
# Example: predict risk for one new/hypothetical employee
sample = X_test.iloc[[0]]  # replace with a manually built row later
sample_scaled = scaler.transform(sample)
prob = model.predict_proba(sample_scaled)[0][1]
print(f"Predicted attrition risk: {prob:.1%}")